# **Aula 3**: Modelagem analítica (OLTP -> OLAP)

OBJETIVOS:

- Partir de dados transacionais reais (OLTP)
- Definir a granularidade correta da análise
- Construir uma Tabela Fato
- Construir Tabelas Dimensão
- Integrar tudo em um Star Schema
- Realizar consultas analíticas (OLAP)
- Preparar os dados para Machine Learning

Dataset: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [0]:
import pandas as pd
import numpy as np
import os

#Célula em Markdown, passe para Code para executar a linha abaixo

#Uma maneira de lidar com datasets Kaggle sem precisar fazer o download

!pip install kagglehub

In [0]:
!pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

## Leitura dos arquivos CSV extraídos do Kaggle (Olist)

Cada arquivo representa uma parte do sistema transacional:
- orders: pedidos
- items: itens vendidos em cada pedido
- customers: dados dos clientes
- products: dados dos produtos

IMPORTANTE:
- Esses dados NÃO estão prontos para análise.
- Eles foram modelados para operação (OLTP).

In [0]:
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))

## Antes de qualquer modelagem analítica, precisamos entender:
- Volume de dados
- Tipos das colunas
- Possíveis valores ausentes

 Essa inspeção é essencial para definir a granularidade e evitar erros futuros no Data Warehouse.

In [0]:
orders.info()

In [0]:
items.info()

In [0]:
customers.info()

In [0]:
products.info()

## GRANULARIDADE DA TABELA FATO

Decisão arquitetural mais importante do Data Warehouse.

 -> 1 linha da Tabela Fato = 1 item vendido em 1 pedido

 Essa granularidade permite:
- Análise detalhada de vendas
- Agregações flexíveis (por cliente, produto, tempo)
- Uso futuro em Machine Learning

In [0]:
items[['order_id', 'order_item_id']].head()

In [0]:
# Quantidade de registros no nível de item
items.shape

## CONSTRUÇÃO DA TABELA FATO

Unimos:
- Itens vendidos (items)
- Informações do pedido (orders)

Esse join ainda representa um modelo operacional, mas já começa a formar a Tabela Fato analítica.

In [0]:
fato = items.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

fato.head()

Selecionamos apenas:
- Métricas numéricas (price, freight_value)
- Chaves estrangeiras (IDs)
- A dimensão temporal (timestamp)

Tudo que NÃO for métrica ou chave deve ir para Tabelas Dimensão.

In [0]:
fato = fato[[
    'order_id',
    'order_item_id',
    'product_id',
    'customer_id',
    'order_purchase_timestamp',
    'price',
    'freight_value'
]]

# Verificando valores ausentes
fato.isna().sum()

In [0]:
fato.head()

Datas devem ser tratadas como datetime para permitir:
- Extração de ano, mês, dia
- Análises de sazonalidade
- Criação da Dimensão Tempo

In [0]:
fato['order_purchase_timestamp'] = pd.to_datetime(
    fato['order_purchase_timestamp']
)

fato.dtypes

In [0]:
fato.head()

## DIMENSÃO CLIENTE

Contém informações qualitativas:
- Estado
- Cidade

Essas variáveis serão usadas como FEATURES em análises e modelos de Machine Learning.

In [0]:
dim_cliente = customers[[
    'customer_id',
    'customer_state',
    'customer_city'
]].drop_duplicates()

dim_cliente.shape

In [0]:
dim_cliente.head()

## DIMENSÃO PRODUTO

Além da categoria, usamos atributos físicos para criar novas FEATURES analíticas.

In [0]:
dim_produto = products[[
    'product_id',
    'product_category_name',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]].drop_duplicates()

dim_produto.head()

## ENGENHARIA DE FEATURES

Criamos uma variável derivada:
-> Volume do produto (cm³)

Isso mostra que o Data Warehouse não é apenas armazenamento, mas também preparação para ML.

In [0]:
dim_produto['product_volume_cm3'] = (
    dim_produto['product_length_cm'] *
    dim_produto['product_height_cm'] *
    dim_produto['product_width_cm']
)

dim_produto[['product_category_name', 'product_volume_cm3']].head()

## DIMENSÃO TEMPO

Fundamental para qualquer análise histórica.

Permite:
- Tendências
- Sazonalidade
- Comparações temporais

In [0]:
dim_tempo = pd.DataFrame({
    'data': fato['order_purchase_timestamp'].dt.date
}).drop_duplicates()

dim_tempo['ano'] = pd.to_datetime(dim_tempo['data']).dt.year
dim_tempo['mes'] = pd.to_datetime(dim_tempo['data']).dt.month
dim_tempo['dia'] = pd.to_datetime(dim_tempo['data']).dt.day
dim_tempo['dia_semana'] = pd.to_datetime(dim_tempo['data']).dt.day_name()
dim_tempo['fim_de_semana'] = dim_tempo['dia_semana'].isin(
    ['Saturday', 'Sunday']
)

dim_tempo.head()

## INTEGRAÇÃO FINAL – STAR SCHEMA

Aqui construímos o Data Warehouse Analítico.

A Tabela Fato fica no centro 

As Dimensões entram via chaves estrangeiras

Esse formato é otimizado para OLAP, BI e Machine Learning.

In [0]:
dw = fato.merge(dim_cliente, on='customer_id', how='left')
dw = dw.merge(dim_produto, on='product_id', how='left')
dw = dw.merge(
    dim_tempo,
    left_on=fato['order_purchase_timestamp'].dt.date,
    right_on='data',
    how='left'
)

dw.shape

In [0]:
dw.head()

## VALIDAÇÃO DO DATA WAREHOUSE

Verificações básicas:
- Duplicidades inesperadas
- Valores ausentes após os joins

In [0]:
dw.duplicated().sum()

In [0]:
dw.isna().sum()

## CONSULTAS ANALÍTICAS (OLAP)

Essas consultas seriam lentas em bancos OLTP, mas são rápidas no modelo dimensional.

In [0]:
import time

# 1. Faturamento por categoria
start = time.time()
faturamento_categoria = (
    dw.groupby('product_category_name')['price']
      .sum()
      .sort_values(ascending=False)
)
end = time.time()
OLAP1 = end - start

print("\nFaturamento por categoria (top 5):")
print(faturamento_categoria.head())
print("Tempo de execução (DW):", end - start, "segundos")


# 2. Faturamento por estado
start = time.time()
faturamento_estado = (
    dw.groupby('customer_state')['price']
      .sum()
      .sort_values(ascending=False)
)
end = time.time()
OLAP2 = end - start

print("\nFaturamento por estado:")
print(faturamento_estado.head())
print("Tempo de execução (DW):", end - start, "segundos")


# 3. Faturamento mensal
start = time.time()
faturamento_mensal = dw.groupby('mes')['price'].sum()
end = time.time()
OLAP3 = end - start

print("\nFaturamento por mês:")
print(faturamento_mensal)
print("Tempo de execução (DW):", end - start, "segundos")

## CONSULTAS NO FORMATO OLTP (FORMA ANTIGA / NÃO OTIMIZADA)

Agora vamos responder às MESMAS perguntas, mas partindo dos dados transacionais originais.

In [0]:
# 1. Faturamento por categoria (modo OLTP)
start = time.time()

faturamento_categoria_oltp = (
    items
    .merge(products, on='product_id', how='left')
    .groupby('product_category_name')['price']
    .sum()
    .sort_values(ascending=False)
)

end = time.time()
OLTP1 = end - start

print("Faturamento por categoria (OLTP - top 5):")
print(faturamento_categoria_oltp.head())
print("Tempo de execução (OLTP):", end - start, "segundos")


# 2. Faturamento por estado (modo OLTP)
start = time.time()

faturamento_estado_oltp = (
    items
    .merge(orders[['order_id', 'customer_id']], on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .groupby('customer_state')['price']
    .sum()
    .sort_values(ascending=False)
)

end = time.time()
OLTP2 = end - start

print("\nFaturamento por estado (OLTP):")
print(faturamento_estado_oltp.head())
print("Tempo de execução (OLTP):", end - start, "segundos")


# 3. Faturamento mensal (modo OLTP)
start = time.time()

faturamento_mensal_oltp = (
    items
    .merge(orders[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')
)

faturamento_mensal_oltp['order_purchase_timestamp'] = pd.to_datetime(
    faturamento_mensal_oltp['order_purchase_timestamp']
)

faturamento_mensal_oltp['mes'] = faturamento_mensal_oltp[
    'order_purchase_timestamp'
].dt.month

faturamento_mensal_oltp = (
    faturamento_mensal_oltp.groupby('mes')['price'].sum()
)

end = time.time()
OLTP3 = end - start


print("\nFaturamento mensal (OLTP):")
print(faturamento_mensal_oltp)
print("Tempo de execução (OLTP):", end - start, "segundos")

## COMPARAÇÃO FINAL

Em bases pequenas a diferença pode parecer sutil, mas em milhões de registros o impacto é enorme.

DW (OLAP)  -> menos joins, mais rápido

OLTP      -> mais joins, mais lento

In [0]:
comparacao = pd.DataFrame({
    'Consulta': [
        'Faturamento por categoria',
        'Faturamento por estado',
        'Faturamento mensal'
    ],
    'Tempo DW (s)': [
        OLAP1,  
        OLAP2,
        OLAP3
    ],
    'Tempo OLTP (s)': [
        OLTP1,
        OLTP2,
        OLTP3
    ]
})

comparacao['Ganho (OLTP / DW)'] = (
    comparacao['Tempo OLTP (s)'] /
    comparacao['Tempo DW (s)']
)

comparacao

## DATASET FINAL PARA MACHINE LEARNING

Aqui selecionamos apenas as FEATURES que serão usadas em modelos preditivos.

In [0]:
dataset_ml = dw[[
    'price',
    'freight_value',
    'product_volume_cm3',
    'customer_state',
    'product_category_name',
    'mes',
    'fim_de_semana'
]]

dataset_ml.head()

In [0]:
print("Dimensões do dataset:", dataset_ml.shape)

# Salvando em formato CSV
dataset_ml.to_csv(
    "dw_olist.csv",
    index=False
)